In [63]:
import re
import nltk
import pandas as pd

from nltk.corpus import stopwords
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split

nltk.download("stopwords")

stemmer = nltk.stem.SnowballStemmer("english")
english_stopwords = set(stopwords.words("english"))
from IPython.display import display


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\adwar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [64]:
newsgroups = fetch_20newsgroups(
    subset="all",
    remove=("headers", "footers", "quotes"),
    random_state=0
)

print("Número de documentos:", len(newsgroups.data))
print("Número de clases:", len(newsgroups.target_names))

Número de documentos: 18846
Número de clases: 20


In [65]:
news_df = pd.DataFrame({
    "text": newsgroups.data,
    "label": newsgroups.target
})

news_df["label_name"] = news_df["label"].map(
    dict(enumerate(newsgroups.target_names))
)

news_df.head()

,text,label,label_name
0,FOR SALE\n\n 1945 King Feature...,6,misc.forsale
1,Earlier today I read an ad for REAL-3D animati...,1,comp.graphics
2,Can someone cite Biblical references to homose...,15,soc.religion.christian
3,My friends and I have a buch of books for sale...,6,misc.forsale
4,\n\nThey are using some technology developed b...,11,sci.crypt


In [66]:
newsgroups.target_names

['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

In [67]:
news_df["label_name"].value_counts().sort_index()

label_name
alt.atheism                 799
comp.graphics               973
comp.os.ms-windows.misc     985
comp.sys.ibm.pc.hardware    982
comp.sys.mac.hardware       963
comp.windows.x              988
misc.forsale                975
rec.autos                   990
rec.motorcycles             996
rec.sport.baseball          994
rec.sport.hockey            999
sci.crypt                   991
sci.electronics             984
sci.med                     990
sci.space                   987
soc.religion.christian      997
talk.politics.guns          910
talk.politics.mideast       940
talk.politics.misc          775
talk.religion.misc          628
Name: count, dtype: int64

### Procesamiento de texto

In [68]:
def preprocess(text):
    """Limpia y procesa un mensaje en inglés."""

    # Remover caracteres especiales
    processed = re.sub(r"\W", " ", str(text))

    # Remover caracteres individuales
    processed = re.sub(r"\s+[a-zA-Z]\s+", " ", processed)

    # Remover números
    processed = re.sub(r"[0-9]+", " ", processed)

    # Simplificar espacios consecutivos
    processed = re.sub(r" +", " ", processed)

    # Convertir a minúsculas
    processed = processed.lower()

    # Remover stopwords y aplicar stemming
    return " ".join(
        stemmer.stem(token)
        for token in processed.split()
        if token not in english_stopwords
    )

In [69]:
news_df["processed_text"] = [
    preprocess(text)
    for text in news_df["text"]
]

### Visualización de los textos procesados

In [ ]:
n_ejemplos = 5

ejemplos = (
    news_df[["text", "processed_text"]]
    .sample(n=n_ejemplos)
    .reset_index(drop=True)
    .copy()
)

ejemplos = ejemplos.rename(columns={
    "text": "Texto original",
    "processed_text": "Texto procesado"
})

display(
    ejemplos.style
    .set_properties(**{
        "white-space": "pre-wrap",
        "text-align": "left",
        "vertical-align": "top",
        "font-size": "14px",
        "padding": "12px"
    })
    .set_table_styles([
        {
            "selector": "table",
            "props": [
                ("width", "100%"),
                ("table-layout", "fixed")
            ]
        },
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("font-size", "15px"),
                ("padding", "12px")
            ]
        }
    ])
)

,Texto original,Texto procesado
0,"I posted about this a while ago but without code excerpts noone was able to help me. The problem is that main_win.win is doing fine, but when I create detail_win.win, it does not receive it's initial expose events until main_win.win receives an event. Here are the relevent calls: main_win.win = XCreateSimpleWindow (mydisplay, DefaultRootWindow(mydisplay), myhint.x, myhint.y, myhint.width, myhint.height, main_win.line_thick, fg, bg); XSetStandardProperties(mydisplay, main_win.win, main_win.text, main_win.text, None, argv, argc, &myhint); main_win.gc = XCreateGC (mydisplay, main_win.win, 0, 0); XMapRaised (mydisplay, detail_win.win); XMapSubwindows (mydisplay, main_win.win); The event mask for main_win is: PPosition | PSize | StructureNotifyMask | ExposureMask| KeyPressMask | EnterWindowMask | LeaveWindowMask; The flags are PPosition | PSize I then create detail_win.win with the following calls (hints has new values): detail_win.win = XCreateSimpleWindow (mydisplay, DefaultRootWindow(mydisplay), myhint.x, myhint.y, myhint.width, myhint.height, detail_win.line_thick, fg, bg); XSetStandardProperties(mydisplay, main_win.win, detail_win.text, detail_win.text, None, argv, argc, &myhint); detail_win.gc = XCreateGC (mydisplay, detail_win.win, 0, 0); XMapRaised (mydisplay, detail_win.win); Event Mask and flags are identical to main_win's flags and event mask. If anybody has any idea why the initial expose events of detail_win.win are not received until main_win.win receives an event I'd love to hear from them. Other that that everything works great so there must be some detail I'm overseeing.",post ago without code excerpt noon abl help problem main_win win fine creat detail_win win receiv initi expos event main_win win receiv event relev call main_win win xcreatesimplewindow mydisplay defaultrootwindow mydisplay myhint myhint myhint width myhint height main_win line_thick fg bg xsetstandardproperti mydisplay main_win win main_win text main_win text none argv argc myhint main_win gc xcreategc mydisplay main_win win xmaprais mydisplay detail_win win xmapsubwindow mydisplay main_win win event mask main_win pposit psize structurenotifymask exposuremask keypressmask enterwindowmask leavewindowmask flag pposit psize creat detail_win win follow call hint new valu detail_win win xcreatesimplewindow mydisplay defaultrootwindow mydisplay myhint myhint myhint width myhint height detail_win line_thick fg bg xsetstandardproperti mydisplay main_win win detail_win text detail_win text none argv argc myhint detail_win gc xcreategc mydisplay detail_win win xmaprais mydisplay detail_win win event mask flag ident main_win flag event mask anybodi idea initi expos event detail_win win receiv main_win win receiv event love hear everyth work great must detail overse
1,"This is turning into 'what's a moonbase good for', and I ought not to post when I've a hundred some odd posts to go, but I would think that the real reason to have a moon base is economic. Since someone with space industry will presumeably have a much larger GNP than they would _without_ space industry, eventually, they will simply be able to afford more stuff.",turn moonbas good ought post hundr odd post go would think real reason moon base econom sinc someon space industri presum much larger gnp would _without_ space industri eventu simpli abl afford stuff
2,"From article <1r3jl5$igh@function.mps.ohio-state.edu>, by nevai@mps.ohio-state.edu (Paul Nevai): Well, I don't exaclty know what _should_ be done, but what I do is keep my cpu on and turn my monitor off when not in use. I do this as much for easing power consumption as anything though. Turning off the monitor when not in use has the advantage of requiring less RAM than a screen saver (but it requires more of MY memory to remember to turn it off... pretty easy to remember to turn it on though :-) -- --------------------------------------------------------------------- Instrument Approach Procedures Auto

### Particiones 60% - 10% - 30%

In [71]:
# ------------------------------------------------------------
# Primera división:
# 60 % entrenamiento
# 40 % temporal
# ------------------------------------------------------------

train_df, temp_df = train_test_split(
    news_df,
    test_size=0.40,
    random_state=0,
    stratify=news_df["label"]
)


# ------------------------------------------------------------
# Segunda división del 40 % restante:
#
# 25 % de 40 % = 10 % validación
# 75 % de 40 % = 30 % prueba
# ------------------------------------------------------------

test_df, val_df = train_test_split(
    temp_df,
    test_size=0.25,
    random_state=0,
    stratify=temp_df["label"]
)

In [72]:
print("Total:", len(news_df))
print("Entrenamiento:", len(train_df))
print("Validación:", len(val_df))
print("Prueba:", len(test_df))

print("Entrenamiento:", len(train_df) / len(news_df))
print("Validación:", len(val_df) / len(news_df))
print("Prueba:", len(test_df) / len(news_df))

Total: 18846
Entrenamiento: 11307
Validación: 1885
Prueba: 5654
Entrenamiento: 0.5999681630054123
Validación: 0.10002122466305848
Prueba: 0.30001061233152926


In [73]:
distribucion = pd.DataFrame({
    "Entrenamiento": train_df["label_name"].value_counts(),
    "Validación": val_df["label_name"].value_counts(),
    "Prueba": test_df["label_name"].value_counts()
})

# Mantener el orden original de las 20 categorías
distribucion = distribucion.reindex(newsgroups.target_names)


In [74]:
distribucion.loc["TOTAL"] = distribucion.sum()

distribucion

,Entrenamiento,Validación,Prueba
label_name,,,
alt.atheism,479,80,240
comp.graphics,584,97,292
comp.os.ms-windows.misc,591,98,296
comp.sys.ibm.pc.hardware,589,98,295
comp.sys.mac.hardware,578,96,289
comp.windows.x,593,99,296
misc.forsale,585,97,293
rec.autos,594,99,297
rec.motorcycles,598,100,298


### Modelos y representaciones.